# LAB 10 — Tokenization của QA, offset mapping và answer span alignment
Dataset cho đáp án theo **vị trí ký tự** (`answer_start`), còn Transformer làm việc trên **vị trí token**. Notebook này đi qua: input của QA, `offset_mapping`, chuyển character span → token span, truncation và sliding window. Tokenizer chính: `distilbert-base-uncased`.

In [1]:
import numpy as np, pandas as pd, inspect, warnings
from transformers import AutoTokenizer, DistilBertModel, BertModel
from datasets import load_dataset
from qa_model import char_to_token_span, prepare_train_features, normalize_answer
warnings.filterwarnings("ignore")
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
q, c = "What is the capital of France?", "The capital of France is Paris."

C:\Users\dinhb\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 10. Tokenization của QA — `[CLS] question [SEP] context [SEP]`

In [2]:
enc = tok(q, c, return_offsets_mapping=True)
display(pd.DataFrame({"token": tok.convert_ids_to_tokens(enc["input_ids"]), "input_id": enc["input_ids"], "attention_mask": enc["attention_mask"],
                      "token_type_id": enc["token_type_ids"], "sequence_id": enc.sequence_ids(), "offset (ký tự)": enc["offset_mapping"]}).T)
for name, cls in (("distilbert-base-uncased", DistilBertModel), ("bert-base-uncased", BertModel)):
    print(f"{name}: tokenizer trả về {list(AutoTokenizer.from_pretrained(name)(q, c).keys())} | model.forward nhận token_type_ids? {'token_type_ids' in inspect.signature(cls.forward).parameters}")
r = AutoTokenizer.from_pretrained("roberta-base"); er = r(q, c)
print("roberta-base: tokenizer trả về", list(er.keys()), "|", " ".join(r.convert_ids_to_tokens(er["input_ids"])))

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
token,[CLS],what,is,the,capital,of,france,?,[SEP],the,capital,of,france,is,paris,.,[SEP]
input_id,101,2054,2003,1996,3007,1997,2605,1029,102,1996,3007,1997,2605,2003,3000,1012,102
attention_mask,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
token_type_id,0,0,0,0,0,0,0,0,0,1,1,1,1,1,1,1,1
sequence_id,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN
offset (ký tự),"(0, 0)","(0, 4)","(5, 7)","(8, 11)","(12, 19)","(20, 22)","(23, 29)","(29, 30)","(0, 0)","(0, 3)","(4, 11)","(12, 14)","(15, 21)","(22, 24)","(25, 30)","(30, 31)","(0, 0)"


distilbert-base-uncased: tokenizer trả về ['input_ids', 'token_type_ids', 'attention_mask'] | model.forward nhận token_type_ids? False


bert-base-uncased: tokenizer trả về ['input_ids', 'token_type_ids', 'attention_mask'] | model.forward nhận token_type_ids? True


roberta-base: tokenizer trả về ['input_ids', 'attention_mask'] | <s> What Ġis Ġthe Ġcapital Ġof ĠFrance ? </s> </s> The Ġcapital Ġof ĠFrance Ġis ĠParis . </s>


- `token_type_ids` (segment id): 0 cho câu hỏi, 1 cho context. **BERT dùng**, còn **DistilBERT không nhận** tham số này dù tokenizer vẫn trả về, nên phải bỏ đi trước khi đưa vào model. **RoBERTa không có** token_type_ids, ngăn hai đoạn bằng `</s></s>`. Vì vậy dùng `sequence_ids()` của tokenizer để biết token nào thuộc context, không dựa vào token_type_ids.
- `offset_mapping` cho mỗi token cặp (ký tự đầu, ký tự sau cùng) **trong chuỗi gốc của đoạn tương ứng**: offset của câu hỏi tính trong câu hỏi, của context tính trong context; special token là (0, 0).

## 12. Character index vs word index vs token index

In [3]:
s_char = c.index("Paris"); e_char = s_char + len("Paris")
st, en = char_to_token_span(enc["offset_mapping"], enc.sequence_ids(), s_char, e_char)
print(f'"Paris": character span [{s_char}, {e_char}) (ký tự đầu {s_char}, ký tự cuối {e_char - 1}) | word index trong context = {c.split().index("Paris.")} | token index trong input = {st}..{en}')
print("kiểm tra: context[offset] =", repr(c[enc["offset_mapping"][st][0]:enc["offset_mapping"][en][1]]))
s2 = c.index("capital of France"); st2, en2 = char_to_token_span(enc["offset_mapping"], enc.sequence_ids(), s2, s2 + len("capital of France"))
print(f'"capital of France": ký tự [{s2}, {s2 + 17}) → token {st2}..{en2} =', tok.convert_ids_to_tokens(enc["input_ids"][st2:en2 + 1]))

"Paris": character span [25, 30) (ký tự đầu 25, ký tự cuối 29) | word index trong context = 5 | token index trong input = 14..14
kiểm tra: context[offset] = 'Paris'
"capital of France": ký tự [4, 21) → token 10..12 = ['capital', 'of', 'france']


## 16. Answer span alignment — các trường hợp khó

In [4]:
cases = [("answer bắt đầu giữa token", "Beyoncé performed at the Super Bowl halftime show.", "oncé", None),
         ("answer kết thúc giữa token", "The Normans were descendants of Norse raiders.", "Norma", None),
         ("từ được tách nhiều subword", "Tesla filed patents on alternating current motors.", "alternating current", None),
         ("answer bị truncation (max_length=24)", "The tournament was hosted in many cities. " * 3 + "The final was played in Rome.", "Rome", 24)]
rows = []
for name, ctx, ans, L in cases:
    e = tok("Where?", ctx, return_offsets_mapping=True, truncation="only_second", max_length=L or 512)
    s = ctx.index(ans); st, en = char_to_token_span(e["offset_mapping"], e.sequence_ids(), s, s + len(ans))
    got = ctx[e["offset_mapping"][st][0]:e["offset_mapping"][en][1]] if st else "→ (0, 0) = [CLS]"
    rows.append({"trường hợp": name, "answer": ans, "token span": (st, en), "tokens": tok.convert_ids_to_tokens(e["input_ids"][st:en + 1]) if st else [], "trích lại từ offset": got})
display(pd.DataFrame(rows).set_index("trường hợp"))

,answer,token span,tokens,trích lại từ offset
trường hợp,,,,
answer bắt đầu giữa token,oncé,"(4, 4)",[beyonce],Beyoncé
answer kết thúc giữa token,Norma,"(5, 5)",[norman],Norman
từ được tách nhiều subword,alternating current,"(8, 9)","[alternating, current]",alternating current
answer bị truncation (max_length=24),Rome,"(0, 0)",[],"→ (0, 0) = [CLS]"


- Answer bắt đầu hoặc kết thúc **giữa** một token → lấy cả token chứa ký tự đó. Câu trả lời trích lại có thể dài hơn gold vài ký tự ("Beyoncé" thay vì "oncé"); trong SQuAD thật trường hợp này rất hiếm.
- Tokenizer *uncased* đổi "Beyoncé" thành "beyonce" (bỏ dấu, chữ thường), nhưng **offset vẫn trỏ về chuỗi gốc**, nên câu trả lời trích ra giữ nguyên "Beyoncé". Đây là lý do phải trích bằng offset chứ không decode token.
- Answer **bị truncation / nằm ngoài window** → nhãn (0, 0), tức là trỏ vào [CLS] ("window này không chứa đáp án").

## 13 + 15. Truncation và sliding window trên một context SQuAD thật
`truncation="only_second"` chỉ cắt context (không bao giờ cắt câu hỏi); `stride` = số token chồng lấp giữa hai window liên tiếp; `overflow_to_sample_mapping` cho biết window thuộc câu hỏi nào.

In [5]:
squad = load_dataset("rajpurkar/squad"); dev = squad["validation"]
lens = [len(tok(x)["input_ids"]) for x in dev["context"][:2000]]; i = int(np.argmax(lens)); ex = dev[i]
e = tok(ex["question"], ex["context"], truncation="only_second", max_length=256, stride=64, return_overflowing_tokens=True, return_offsets_mapping=True)
print(f"context dài {lens[i]} token | max_length=256, stride=64 → {len(e['input_ids'])} window | overflow_to_sample_mapping = {e['overflow_to_sample_mapping']}")
s = ex["answers"]["answer_start"][0]; t = ex["answers"]["text"][0]; print("question:", ex["question"], "| gold:", t, f"| ký tự [{s}, {s + len(t)})")
for k in range(len(e["input_ids"])):
    ctx = [j for j, sq in enumerate(e.sequence_ids(k)) if sq == 1]; a, b = e["offset_mapping"][k][ctx[0]][0], e["offset_mapping"][k][ctx[-1]][1]
    st, en = char_to_token_span(e["offset_mapping"][k], e.sequence_ids(k), s, s + len(t))
    print(f"  window {k}: context ký tự [{a}, {b}) | {len(e['input_ids'][k])} token | nhãn start/end = ({st}, {en})" + (f" → {ex['context'][e['offset_mapping'][k][st][0]:e['offset_mapping'][k][en][1]]!r}" if st else ""))
if len(e["input_ids"]) > 1:
    ov = [j for j, sq in enumerate(e.sequence_ids(1)) if sq == 1][:64]; print("  64 token context đầu của window 1 trùng với cuối window 0 (stride):", tok.decode([e["input_ids"][1][j] for j in ov])[:120], "...")

context dài 416 token | max_length=256, stride=64 → 2 window | overflow_to_sample_mapping = [0, 0]
question: Who is the head coach of the Broncos? | gold: Gary Kubiak | ký tự [291, 302)
  window 0: context ký tự [0, 1181) | 256 token | nhãn start/end = (64, 67) → 'Gary Kubiak'
  window 1: context ký tự [891, 2060) | 246 token | nhãn start/end = (0, 0)
  64 token context đầu của window 1 trùng với cuối window 0 (stride): the nfl in interceptions. in week 10, manning suffered a partial tear of the plantar fasciitis in his left foot. he set  ...


Context dài 416 token với max_length 256 → 2 window chồng lấp nhau 64 token. Đáp án "Gary Kubiak" (ký tự 291–301) nằm trong window 0 nên window 0 có nhãn (64, 67); window 1 không chứa đáp án nên nhãn (0, 0) = [CLS]. Chỉ số token 64 tính trong **window** (gồm cả câu hỏi), không phải trong context.

## Thống kê trên SQuAD train (20,000 câu hỏi đầu): mất bao nhiêu đáp án nếu chỉ truncation, và alignment có khôi phục đúng không

In [6]:
tr = squad["train"].select(range(20000)); Q, Cx, A = list(tr["question"]), list(tr["context"]), list(tr["answers"]); rows = []
for L, stride in ((128, 42), (256, 85), (384, 128)):
    plain = tok([x.lstrip() for x in Q], Cx, truncation="only_second", max_length=L, return_offsets_mapping=True)
    lost = sum(char_to_token_span(plain["offset_mapping"][k], plain.sequence_ids(k), A[k]["answer_start"][0], A[k]["answer_start"][0] + len(A[k]["text"][0]))[0] == 0 for k in range(len(Q)))
    f = prepare_train_features(tok, Q, Cx, A, L, stride); covered = len({f["overflow_to_sample_mapping"][k] for k in range(len(f["input_ids"])) if f["start_positions"][k] != 0})
    ok = n = 0; bad = []
    for k in range(len(f["input_ids"])):
        st, en = f["start_positions"][k], f["end_positions"][k]
        if st == 0: continue
        ex_ = f["overflow_to_sample_mapping"][k]; got = Cx[ex_][f["offset_mapping"][k][st][0]:f["offset_mapping"][k][en][1]]; n += 1
        if normalize_answer(got) == normalize_answer(A[ex_]["text"][0]): ok += 1
        elif len(bad) < 3: bad.append((A[ex_]["text"][0], got))
    rows.append({"max_length": L, "stride": stride, "% đáp án mất khi chỉ truncate": round(lost / len(Q) * 100, 2), "% câu hỏi có ≥1 window chứa đáp án (sliding)": round(covered / len(Q) * 100, 2),
                 "số window": len(f["input_ids"]), "% window khôi phục đúng đáp án": round(ok / n * 100, 2)})
display(pd.DataFrame(rows).set_index("max_length")); print("ví dụ không khôi phục được (gold, trích lại):", bad)

,stride,% đáp án mất khi chỉ truncate,% câu hỏi có ≥1 window chứa đáp án (sliding),số window,% window khôi phục đúng đáp án
max_length,,,,,
128,42,13.54,99.99,36885,99.30
256,85,0.91,100.00,21717,99.27
384,128,0.07,100.00,20164,99.26


ví dụ không khôi phục được (gold, trích lại): [('split with Luckett and Rober', 'split with Luckett and Roberson'), ('six', 'sixth'), ('evolution', 'evolutionary')]


**Nhận xét:**
- **Chỉ truncation** (không sliding window): max_length 128 làm **mất 13.5%** đáp án (bị cắt khỏi window), 256 mất 0.9%, 384 mất 0.07%.
- **Sliding window** giữ được đáp án cho gần **100%** câu hỏi ở mọi độ dài, đổi lại số window tăng: 20,164 window (384) → 36,885 (128), tức nhiều phép tính hơn khi train và dự đoán.
- Alignment khôi phục đúng đáp án **99.3%**. Phần còn lại là gold **được gán giữa một từ** ("Rober" trong "Roberson", "six" trong "sixth", "evolution" trong "evolutionary"): token nhỏ nhất là cả từ, nên model không bao giờ trích ra đúng chuỗi đó được. Đây là trần EM khoảng 99% do cách gán nhãn, không phải lỗi alignment.